# Distance-Aware Graph Attention Network for Protein Stability Prediction

This notebook implements the complete pipeline for predicting changes in protein
thermodynamic stability (DDG) upon single-point mutations using a distance-aware
graph attention mechanism on protein contact networks.

**Training data:** S8754 (8,754 mutations from 301 proteins)  
**Evaluation:** S669 blind test set (669 mutations, <25% seq identity to training)  
**Input files:** `S8754.csv`, `S669.csv`

In [ ]:
# Install required packages
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.5.0+cu121.html -q
!pip install transformers -q

import os, json, time, copy, gc, requests, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.utils import softmax
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import cdist
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- Paths ---
BASE = "/kaggle/working"
RAW_DIR = "/kaggle/input/datasets/akhurathganapathy/protein-stability-data"
RESULTS_DIR = "/kaggle/working/results_v4"
PDB_DIR = "/kaggle/working/pdb_cache"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PDB_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# --- Constants ---
EDGE_CUTOFF = 12.0                      # Angstroms; all residue pairs within this distance get an edge
DDG_CLIP_MIN, DDG_CLIP_MAX = -8.0, 6.0  # Clip training outliers to this range

AA_3TO1 = {'ALA':'A','CYS':'C','ASP':'D','GLU':'E','PHE':'F','GLY':'G',
           'HIS':'H','ILE':'I','LYS':'K','LEU':'L','MET':'M','ASN':'N',
           'PRO':'P','GLN':'Q','ARG':'R','SER':'S','THR':'T','VAL':'V',
           'TRP':'W','TYR':'Y','MSE':'M'}
AA_LIST = list('ACDEFGHIKLMNPQRSTVWY')
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_LIST)}

# Physicochemical property tables for mutation encoding
HYDRO = {'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,'H':-3.2,
         'I':4.5,'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,
         'R':-4.5,'S':-0.8,'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3}
VOLUME = {'A':88.6,'C':108.5,'D':111.1,'E':138.4,'F':189.9,'G':60.1,'H':153.2,
          'I':166.7,'K':168.6,'L':166.7,'M':162.9,'N':114.1,'P':112.7,'Q':143.8,
          'R':173.4,'S':89.0,'T':116.1,'V':140.0,'W':227.8,'Y':193.6}
CHARGE = {'D':-1,'E':-1,'K':1,'R':1,'H':0.5}

print("Setup complete.")

In [ ]:
# --- Load datasets and extract ESM-2 embeddings ---

train_df = pd.read_csv(os.path.join(RAW_DIR, 'S8754.csv'))
test_df = pd.read_csv(os.path.join(RAW_DIR, 'S669.csv'))

# Clip extreme DDG outliers in the training set.
# The test set ranges from -8.93 to 5.73, but training has values as extreme as -17.4 and +13.7.
# These outliers distort the loss and are not representative of the test distribution.
n_clipped = ((train_df['ddG'] < DDG_CLIP_MIN) | (train_df['ddG'] > DDG_CLIP_MAX)).sum()
train_df['ddG'] = train_df['ddG'].clip(DDG_CLIP_MIN, DDG_CLIP_MAX)
print(f"Train: {len(train_df)} mutations ({n_clipped} DDG values clipped to [{DDG_CLIP_MIN}, {DDG_CLIP_MAX}])")
print(f"Test: {len(test_df)} mutations (unchanged)")

# Collect all unique sequences. We need embeddings for both wild-type and mutant
# sequences so we can compute the delta embedding at the mutation site.
all_seqs = set()
for df in [train_df, test_df]:
    all_seqs.update(df['wt_seq'].tolist())
    all_seqs.update(df['mut_seq'].tolist())
print(f"Unique sequences (WT + MUT): {len(all_seqs)}")

# Load the ESM-2 protein language model (35M parameters, 480-dimensional output).
# This is the mid-size variant; larger models exist but are slower.
from transformers import EsmTokenizer, EsmModel
ESM_NAME = "facebook/esm2_t12_35M_UR50D"
ESM_DIM = 480
print(f"\nLoading {ESM_NAME} ({ESM_DIM}D per residue)...")
esm_tok = EsmTokenizer.from_pretrained(ESM_NAME)
esm_mdl = EsmModel.from_pretrained(ESM_NAME).to(device)
esm_mdl.eval()
print("Model loaded.")

# Extract per-residue embeddings for every unique sequence.
# BOS and EOS tokens are removed so position i maps directly to residue i.
seq2emb = {}
print(f"\nExtracting embeddings for {len(all_seqs)} sequences...")
t0 = time.time()
with torch.no_grad():
    for i, seq in enumerate(sorted(all_seqs)):
        inp = esm_tok(seq[:1022], return_tensors='pt', padding=False,
                      truncation=True, max_length=1024).to(device)
        out = esm_mdl(**inp)
        seq2emb[seq] = out.last_hidden_state[0, 1:-1, :].cpu()
        if (i+1) % 100 == 0:
            eta = (time.time()-t0)/(i+1)*(len(all_seqs)-i-1)/60
            print(f"  [{i+1}/{len(all_seqs)}] ETA: {eta:.1f} min")

print(f"Finished in {(time.time()-t0)/60:.1f} min")

# Free the ESM model from GPU memory before training our own model
del esm_mdl, esm_tok
torch.cuda.empty_cache()
gc.collect()
print("ESM-2 model released from memory.")

In [ ]:
# --- Graph construction functions ---
#
# Each mutation is represented as a protein contact network (graph) where:
#   Nodes  = residues, with features from ESM-2 + mutation encoding
#   Edges  = all Ca-Ca pairs within 12 Angstroms
#   Labels = experimental DDG

def download_pdb(pdb_id):
    """Download a PDB file from RCSB if not already cached."""
    path = os.path.join(PDB_DIR, f"{pdb_id}.pdb")
    if os.path.exists(path):
        return path
    try:
        r = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30)
        if r.status_code == 200:
            with open(path, 'w') as f:
                f.write(r.text)
            return path
    except Exception:
        pass
    return None


def get_ca(pdb_path, chain):
    """Extract alpha-carbon coordinates and residue identities from a PDB chain."""
    residues = []
    seen = set()
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith('ATOM') or line[12:16].strip() != 'CA':
                continue
            if line[21].strip() != chain:
                continue
            resseq = int(line[22:26].strip())
            icode = line[26].strip()
            if (resseq, icode) in seen:
                continue
            seen.add((resseq, icode))
            aa = AA_3TO1.get(line[17:20].strip(), 'X')
            if aa == 'X':
                continue
            coords = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            residues.append({'aa': aa, 'coords': coords})
    return residues


def align(pdb_res, csv_seq):
    """Align PDB residues to the CSV sequence using substring matching.
    
    PDB residue numbering often differs from sequence indexing, so we
    match the PDB-derived sequence against the CSV sequence to find the
    correct offset. Falls back to a sliding-window approach if an exact
    match is not found.
    """
    pdb_seq = ''.join(r['aa'] for r in pdb_res)
    # Try exact substring match first
    idx = csv_seq.find(pdb_seq)
    if idx >= 0:
        return [(idx + i, pdb_res[i]) for i in range(len(pdb_res))]
    # Sliding window: find the offset with the most character matches
    best_offset, best_count = 0, 0
    for offset in range(max(0, len(csv_seq) - len(pdb_seq) + 1)):
        count = sum(1 for i in range(min(len(pdb_seq), len(csv_seq) - offset))
                    if pdb_seq[i] == csv_seq[offset + i])
        if count > best_count:
            best_count, best_offset = count, offset
    if best_count >= len(pdb_seq) * 0.8:
        return [(best_offset + i, pdb_res[i])
                for i in range(min(len(pdb_seq), len(csv_seq) - best_offset))
                if pdb_seq[i] == csv_seq[best_offset + i]]
    return None


def build_graph_v4(row, seq2emb):
    """Construct a PyG Data object for a single mutation.
    
    Node features (970D per residue):
        - Amino acid index (1)       -> passed through learned embedding during forward
        - Mutation site flag (1)
        - WT and MUT amino acid indices (2)
        - Spatial distance to mutation site (1)
        - pH and temperature (2)
        - Delta hydrophobicity, volume, charge (3)
        - ESM-2 WT embedding at this position (480)
        - ESM-2 delta embedding at mutation site, broadcast to all nodes (480)
    
    Edge features (4D per edge):
        - Normalised Ca-Ca distance (1)
        - Unit direction vector (3)
    """
    parts = row['name'].split('_')
    if parts[0] != 'rcsb' or len(parts) < 6:
        return None
    pdb_id, chain = parts[1].upper(), parts[2]
    wt_aa, mut_aa = parts[3][0], parts[3][-1]
    wt_seq, mut_seq = row['wt_seq'], row['mut_seq']
    ddG = row['ddG']
    pH = float(parts[4]) if len(parts) > 4 else 7.0
    temp = float(parts[5]) if len(parts) > 5 else 25.0

    # Identify mutation position by comparing WT and MUT sequences directly
    mut_seq_idx = None
    for i in range(min(len(wt_seq), len(mut_seq))):
        if wt_seq[i] != mut_seq[i]:
            mut_seq_idx = i
            break
    if mut_seq_idx is None:
        return None

    # Obtain and align the PDB structure
    pdb_path = download_pdb(pdb_id)
    if not pdb_path:
        return None
    pdb_res = get_ca(pdb_path, chain)
    if len(pdb_res) < 10:
        return None
    alignment = align(pdb_res, wt_seq)
    if not alignment or len(alignment) < 10:
        return None
    if wt_seq not in seq2emb or mut_seq not in seq2emb:
        return None

    wt_emb = seq2emb[wt_seq]
    mut_emb = seq2emb[mut_seq]

    # Locate the mutation node within the aligned residue set
    all_coords = np.array([a[1]['coords'] for a in alignment])
    mut_node = None
    for ni, (si, _) in enumerate(alignment):
        if si == mut_seq_idx:
            mut_node = ni
            break
    if mut_node is None:
        return None

    n = len(alignment)
    mut_coords = all_coords[mut_node]
    dists_to_mut = np.linalg.norm(all_coords - mut_coords, axis=1)

    # Physicochemical deltas for the specific substitution
    delta_hydro = (HYDRO.get(mut_aa, 0) - HYDRO.get(wt_aa, 0)) / 9.0
    delta_vol = (VOLUME.get(mut_aa, 130) - VOLUME.get(wt_aa, 130)) / 150.0
    delta_charge = CHARGE.get(mut_aa, 0) - CHARGE.get(wt_aa, 0)

    # ESM-2 delta: difference between mutant and wild-type embeddings at the mutation site.
    # This is broadcast to all nodes so the model knows what changed regardless of
    # which node it is currently processing.
    if mut_seq_idx < wt_emb.shape[0] and mut_seq_idx < mut_emb.shape[0]:
        esm_delta_at_mut = (mut_emb[mut_seq_idx] - wt_emb[mut_seq_idx]).numpy()
    else:
        esm_delta_at_mut = np.zeros(ESM_DIM)

    # Assemble node features
    node_feats = []
    for ni, (si, pr) in enumerate(alignment):
        aa_idx = AA_TO_IDX.get(pr['aa'], 0)
        is_mut = 1.0 if ni == mut_node else 0.0
        d_to_mut = dists_to_mut[ni] / 20.0
        wt_esm = wt_emb[si].numpy() if si < wt_emb.shape[0] else np.zeros(ESM_DIM)

        feat = np.concatenate([
            [aa_idx, is_mut, AA_TO_IDX.get(wt_aa, 0), AA_TO_IDX.get(mut_aa, 0),
             d_to_mut, pH / 14.0, temp / 100.0, delta_hydro, delta_vol, delta_charge],
            wt_esm,
            esm_delta_at_mut,
        ])
        node_feats.append(feat)
    node_feats = np.array(node_feats, dtype=np.float32)

    # Build edges between all Ca pairs within the cutoff
    dm = cdist(all_coords, all_coords)
    src, dst, ed, edir = [], [], [], []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            d = dm[i, j]
            if d <= EDGE_CUTOFF:
                src.append(i)
                dst.append(j)
                ed.append(d / EDGE_CUTOFF)
                edir.append((all_coords[j] - all_coords[i]) / (d + 1e-8))
    if not src:
        return None

    ei = torch.tensor([src, dst], dtype=torch.long)
    e_d = torch.tensor(ed, dtype=torch.float32)
    ea = torch.cat([e_d.unsqueeze(-1), torch.tensor(np.array(edir), dtype=torch.float32)], 1)

    g = Data(x=torch.tensor(node_feats, dtype=torch.float32),
             edge_index=ei, edge_attr=ea, edge_dist=e_d,
             y=torch.tensor([ddG], dtype=torch.float32),
             mut_idx=torch.tensor([mut_node], dtype=torch.long),
             pos=torch.tensor(all_coords, dtype=torch.float32))
    g.pdb_id = pdb_id
    g.chain = chain
    return g


print("Graph construction functions defined.")

In [ ]:
# --- Build all graphs and prepare train/val split ---

def build_all(df, name):
    graphs, failed = [], 0
    t0 = time.time()
    for i, (_, row) in enumerate(df.iterrows()):
        g = build_graph_v4(row, seq2emb)
        if g is not None:
            graphs.append(g)
        else:
            failed += 1
        if (i + 1) % 500 == 0:
            eta = (time.time() - t0) / (i + 1) * (len(df) - i - 1) / 60
            print(f"  [{i+1}/{len(df)}] built={len(graphs)} failed={failed} ETA={eta:.1f} min")
    elapsed = (time.time() - t0) / 60
    print(f"  {name}: {len(graphs)} graphs, {failed} failed ({elapsed:.1f} min)")
    return graphs


print("Building training graphs...")
train_graphs = build_all(train_df, 'train')
print("\nBuilding test graphs...")
test_graphs = build_all(test_df, 'test')

# Release ESM embeddings from memory
del seq2emb
gc.collect()

in_ch = train_graphs[0].x.shape[1]
print(f"\nNode feature dimensionality: {in_ch}")

# Compute per-protein sample weights to counteract data imbalance.
# Without weighting, 1PGA alone accounts for 17.6% of training data and
# the top 10 proteins account for 46.6%. This causes the model to overfit
# to heavily-studied proteins rather than learning general stability physics.
from collections import Counter
pdb_counts = Counter(g.pdb_id for g in train_graphs)
for g in train_graphs:
    g.sample_weight = 1.0 / math.sqrt(pdb_counts[g.pdb_id])

print(f"\nPer-protein weighting applied. Examples:")
for pdb, cnt in pdb_counts.most_common(5):
    w = 1.0 / math.sqrt(cnt)
    print(f"  {pdb}: {cnt} mutations, weight = {w:.3f}")

# Protein-level validation split: all mutations from a given protein go
# entirely to either training or validation. This avoids information leakage
# where the model sees the same protein structure during both training and
# validation, which would inflate validation metrics.
train_pdbs = list(set(g.pdb_id for g in train_graphs))
np.random.seed(42)
np.random.shuffle(train_pdbs)
n_val_pdbs = max(5, len(train_pdbs) // 10)
val_pdb_set = set(train_pdbs[:n_val_pdbs])

actual_train = [g for g in train_graphs if g.pdb_id not in val_pdb_set]
val_graphs = [g for g in train_graphs if g.pdb_id in val_pdb_set]

print(f"\nProtein-level validation split:")
print(f"  Training:   {len(actual_train)} mutations from {len(set(g.pdb_id for g in actual_train))} proteins")
print(f"  Validation: {len(val_graphs)} mutations from {len(val_pdb_set)} proteins")
print(f"  Protein overlap between train/val: 0")
print(f"\nTotal: {len(train_graphs)} training + {len(test_graphs)} test graphs.")

In [ ]:
# --- Model definition ---

class DistanceKernel(nn.Module):
    """Mixture-of-Gaussians kernel for learning distance-dependent interaction weights.
    
    Parameterised by M Gaussian components with learnable widths (sigma) and
    mixing weights. The kernel output phi(d) is a weighted sum of Gaussian
    responses at the input distance, representing how strongly two residues
    at distance d should attend to each other.
    """
    def __init__(self, n_kernels=3):
        super().__init__()
        # Initialised at physically motivated scales (normalised by 12A cutoff):
        # 0.10 ~ 1.2A (backbone), 0.25 ~ 3A (H-bonds/vdW), 0.50 ~ 6A (electrostatic)
        self.log_s = nn.Parameter(torch.log(torch.tensor([0.10, 0.25, 0.50][:n_kernels])))
        self.w = nn.Parameter(torch.ones(n_kernels) / n_kernels)
        self.n_k = n_kernels

    def forward(self, d):
        sigma = torch.exp(self.log_s)
        weights = F.softmax(self.w, dim=0)
        gaussians = torch.exp(-d.unsqueeze(-1) ** 2 / (2 * sigma ** 2 + 1e-8))
        return (gaussians * weights).sum(dim=-1)


class DistanceAwareGATLayer(MessagePassing):
    """Graph attention layer with additive distance-aware bias.
    
    Standard GAT computes attention as:
        e_ij = LeakyReLU(a_s[h_i] + a_d[h_j])
    
    This layer adds two additional terms:
        e_ij = LeakyReLU(a_s[h_i] + a_d[h_j] + edge_bias_ij + gamma * phi(d_ij))
    
    where edge_bias comes from a projection of the edge features (distance +
    direction vector) and phi(d) is the learned distance kernel.
    """
    def __init__(self, in_ch, out_ch, heads=4, n_kernels=3, edge_dim=4,
                 dropout=0.2, concat=True, use_distance=True):
        super().__init__(aggr='add', node_dim=0)
        self.heads = heads
        self.out_ch = out_ch
        self.concat = concat
        self.dropout = dropout
        self.use_distance = use_distance

        self.W = nn.Linear(in_ch, heads * out_ch, bias=False)
        self.a_s = nn.Parameter(torch.zeros(1, heads, out_ch))
        self.a_d = nn.Parameter(torch.zeros(1, heads, out_ch))
        self.edge_proj = nn.Sequential(nn.Linear(edge_dim, heads), nn.LeakyReLU(0.2))

        if use_distance:
            self.dk = DistanceKernel(n_kernels)
            self.gamma = nn.Parameter(torch.tensor(1.0))

        out_dim = heads * out_ch if concat else out_ch
        self.ln = nn.LayerNorm(out_dim)
        self.leaky_relu = nn.LeakyReLU(0.2)

        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a_s)
        nn.init.xavier_uniform_(self.a_d)

    def forward(self, x, edge_index, edge_attr, edge_dist):
        H = self.W(x).view(-1, self.heads, self.out_ch)
        self._alpha_src = (H * self.a_s).sum(-1)
        self._alpha_dst = (H * self.a_d).sum(-1)
        self._H = H
        self._edge_bias = self.edge_proj(edge_attr)

        if self.use_distance:
            self._dist_bias = (self.gamma * self.dk(edge_dist)).unsqueeze(-1)
        else:
            self._dist_bias = torch.zeros(edge_dist.shape[0], 1, device=x.device)

        out = self.propagate(edge_index, size=None)
        if self.concat:
            out = out.view(-1, self.heads * self.out_ch)
        else:
            out = out.mean(dim=1)
        return self.ln(out)

    def message(self, edge_index_i, edge_index_j, size_i):
        alpha = (self._alpha_src[edge_index_i]
                 + self._alpha_dst[edge_index_j]
                 + self._edge_bias
                 + self._dist_bias)
        alpha = self.leaky_relu(alpha)
        alpha = softmax(alpha, edge_index_i, num_nodes=size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        return self._H[edge_index_j] * alpha.unsqueeze(-1)

    def aggregate(self, inputs, index, dim_size=None):
        from torch_geometric.utils import scatter
        return scatter(inputs, index, dim=0, dim_size=dim_size, reduce='sum')


class ProteinStabilityModel(nn.Module):
    """Distance-Aware GAT for DDG prediction.
    
    Architecture:
        Input projection (970D -> 256D)
        -> DA-GAT Layer 1 (4 heads x 64, with residual connection)
        -> DA-GAT Layer 2 (4 heads x 64, with residual connection)
        -> Readout: [mutation site node || global mean pool] (512D)
        -> Regression head (512 -> 256 -> 64 -> 1)
    """
    def __init__(self, n_feat=970, hidden=64, heads=4, n_kernels=3, edge_dim=4,
                 dropout=0.3, use_distance=True):
        super().__init__()
        self.dropout = dropout
        self.aa_embedding = nn.Embedding(20, 32)

        input_dim = 32 + (n_feat - 1)  # learned embedding replaces the raw AA index
        gat_dim = heads * hidden        # 256

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, gat_dim),
            nn.LayerNorm(gat_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.gat1 = DistanceAwareGATLayer(gat_dim, hidden, heads, n_kernels, edge_dim,
                                          dropout, concat=True, use_distance=use_distance)
        self.gat2 = DistanceAwareGATLayer(gat_dim, hidden, heads, n_kernels, edge_dim,
                                          dropout, concat=True, use_distance=use_distance)
        self.residual1 = nn.Linear(gat_dim, gat_dim)
        self.residual2 = nn.Linear(gat_dim, gat_dim)

        # Regression head takes the mutation site representation concatenated
        # with a global mean pooling of the full graph.
        self.regressor = nn.Sequential(
            nn.Linear(gat_dim * 2, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1),
        )

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        edge_dist = data.edge_dist
        batch = data.batch
        mut_idx = data.mut_idx

        # Replace the raw AA index with a learned embedding
        aa_embed = self.aa_embedding(x[:, 0].long().clamp(0, 19))
        x = torch.cat([aa_embed, x[:, 1:]], dim=1)

        x = self.input_proj(x)

        # Two GAT layers with residual connections
        x = F.gelu(self.gat1(x, edge_index, edge_attr, edge_dist) + self.residual1(x))
        x = F.dropout(x, self.dropout, self.training)
        x = F.gelu(self.gat2(x, edge_index, edge_attr, edge_dist) + self.residual2(x))
        x = F.dropout(x, self.dropout, self.training)

        # Compute batch offsets for mutation node indexing
        batch_offsets = torch.zeros_like(mut_idx)
        if batch is not None:
            for i in range(1, len(mut_idx)):
                batch_offsets[i] = (batch < i).sum()

        # Extract mutation site and global representations
        mut_repr = x[mut_idx.squeeze(-1) + batch_offsets]
        global_repr = global_mean_pool(x, batch)

        combined = torch.cat([mut_repr, global_repr], dim=1)
        return self.regressor(combined).squeeze(-1)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = ProteinStabilityModel(n_feat=in_ch, use_distance=True).to(device)
print(f"Trainable parameters: {model.count_parameters():,}")

In [ ]:
# --- Training loop ---

CFG = {
    'epochs': 150,
    'bs': 32,
    'lr': 2e-4,
    'wd': 5e-4,
    'patience': 25,
}


def pearson_loss(pred, target):
    """Differentiable Pearson correlation loss (1 - r)."""
    vp = pred - pred.mean()
    vt = target - target.mean()
    r = (vp * vt).sum() / (torch.sqrt((vp ** 2).sum()) * torch.sqrt((vt ** 2).sum()) + 1e-8)
    return 1.0 - r


def weighted_huber_loss(pred, target, weights, delta=2.0):
    """Huber loss weighted by per-sample protein weights."""
    diff = torch.abs(pred - target)
    huber = torch.where(diff < delta, 0.5 * diff ** 2, delta * (diff - 0.5 * delta))
    return (huber * weights).mean()


def train_epoch(model, loader, optimizer, dev):
    model.train()
    total_loss, n_samples = 0, 0
    for batch in loader:
        batch = batch.to(dev)
        optimizer.zero_grad()
        pred = model(batch)
        target = batch.y.squeeze()
        weights = batch.sample_weight if hasattr(batch, 'sample_weight') else torch.ones_like(target)

        loss_huber = weighted_huber_loss(pred, target, weights, delta=2.0)
        loss_corr = pearson_loss(pred, target) if len(target) > 4 else torch.tensor(0.0, device=dev)
        loss = loss_huber + 0.5 * loss_corr

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(target)
        n_samples += len(target)
    return total_loss / n_samples


@torch.no_grad()
def evaluate(model, loader, dev):
    model.eval()
    all_preds, all_targets = [], []
    for batch in loader:
        batch = batch.to(dev)
        all_preds.extend(model(batch).cpu().numpy())
        all_targets.extend(batch.y.squeeze().cpu().numpy())
    preds = np.array(all_preds)
    targets = np.array(all_targets)
    return {
        'pcc': pearsonr(targets, preds)[0],
        'scc': spearmanr(targets, preds)[0],
        'rmse': np.sqrt(mean_squared_error(targets, preds)),
        'mae': mean_absolute_error(targets, preds),
        'preds': preds,
        'targets': targets,
    }


def run_experiment(name, train_data, val_data, test_data, use_distance=True):
    print(f"\n{'=' * 60}")
    print(f"{name}")
    print(f"{'=' * 60}")

    train_loader = DataLoader(train_data, batch_size=CFG['bs'], shuffle=True, drop_last=True)
    val_loader = DataLoader(val_data, batch_size=CFG['bs'])
    test_loader = DataLoader(test_data, batch_size=CFG['bs'])

    m = ProteinStabilityModel(n_feat=train_data[0].x.shape[1],
                              use_distance=use_distance).to(device)
    optimizer = torch.optim.AdamW(m.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=2, eta_min=1e-6)

    best_val_pcc = -1
    patience_counter = 0
    best_state = None
    t0 = time.time()

    for epoch in range(CFG['epochs']):
        train_loss = train_epoch(m, train_loader, optimizer, device)
        val_metrics = evaluate(m, val_loader, device)
        scheduler.step()

        if val_metrics['pcc'] > best_val_pcc:
            best_val_pcc = val_metrics['pcc']
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in m.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= CFG['patience']:
                print(f"  Early stopping at epoch {epoch + 1}")
                break

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}: loss={train_loss:.4f}  "
                  f"val_r={val_metrics['pcc']:.4f}  val_rmse={val_metrics['rmse']:.3f}")

    # Evaluate on the blind test set using the best checkpoint
    m.load_state_dict(best_state)
    m.to(device)
    test_metrics = evaluate(m, test_loader, device)
    elapsed = (time.time() - t0) / 60

    print(f"\n  S669 Blind Test Results:")
    print(f"    Pearson r:  {test_metrics['pcc']:.4f}")
    print(f"    Spearman p: {test_metrics['scc']:.4f}")
    print(f"    RMSE:       {test_metrics['rmse']:.3f} kcal/mol")
    print(f"    MAE:        {test_metrics['mae']:.3f} kcal/mol")
    print(f"    Time:       {elapsed:.1f} min")

    return {**test_metrics, 'name': name, 'state': best_state, 'time': elapsed}


print("Training functions defined.")

In [ ]:
# --- Run experiments ---

results = {}

# Full model with distance-aware attention
results['Full'] = run_experiment(
    'Distance-Aware GAT',
    actual_train, val_graphs, test_graphs, use_distance=True)

# Ablation: identical architecture with distance kernel disabled
results['NoDist'] = run_experiment(
    'Standard GAT (no distance kernel)',
    actual_train, val_graphs, test_graphs, use_distance=False)

# --- Summary table ---
print(f"\n\n{'=' * 75}")
print(f"S669 Blind Test Results")
print(f"{'=' * 75}")
print(f"{'Method':<42} {'Pearson r':>10} {'Spearman':>10} {'RMSE':>8} {'MAE':>8}")
print(f"{'─' * 75}")
for key, r in results.items():
    print(f"{r['name']:<42} {r['pcc']:>10.4f} {r['scc']:>10.4f} {r['rmse']:>8.3f} {r['mae']:>8.3f}")
print(f"{'─' * 75}")
benchmarks = [
    ('DynaMut (published)', 0.415, 1.596),
    ('DDMut (published)', 0.440, 1.492),
    ('ACDC-NN (published)', 0.460, 1.488),
    ('GeoDDG-3D (published)', 0.601, 1.332),
]
for name, r_val, rmse_val in benchmarks:
    print(f"{name:<42} {r_val:>10.3f} {'--':>10} {rmse_val:>8.3f} {'--':>8}")
print(f"{'=' * 75}")

delta_r = results['Full']['pcc'] - results['NoDist']['pcc']
print(f"\nDistance kernel contribution: +{delta_r:.4f}")
print(f"vs DynaMut:  {'+' if results['Full']['pcc'] > 0.415 else ''}{results['Full']['pcc'] - 0.415:.4f}")
print(f"vs DDMut:    {'+' if results['Full']['pcc'] > 0.440 else ''}{results['Full']['pcc'] - 0.440:.4f}")

In [ ]:
# --- Visualisation ---

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel A: Predicted vs experimental DDG
r = results['Full']
ax = axes[0]
ax.scatter(r['targets'], r['preds'], alpha=0.4, s=15, c='steelblue', edgecolors='none')
lims = [min(r['targets'].min(), r['preds'].min()) - 1,
        max(r['targets'].max(), r['preds'].max()) + 1]
ax.plot(lims, lims, 'r--', lw=1, alpha=0.5)
ax.set_xlabel('Experimental DDG (kcal/mol)', fontsize=12)
ax.set_ylabel('Predicted DDG (kcal/mol)', fontsize=12)
ax.set_title(f'DA-GAT: r = {r["pcc"]:.3f}, RMSE = {r["rmse"]:.2f}', fontsize=13, fontweight='bold')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Panel B: Benchmark comparison
ax = axes[1]
method_names = ['DA-GAT\n(Ours)', 'Std GAT\n(no dist)', 'DynaMut', 'DDMut', 'ACDC-NN', 'GeoDDG-3D']
method_r = [results['Full']['pcc'], results['NoDist']['pcc'], 0.415, 0.440, 0.460, 0.601]
bar_colors = ['#27ae60', '#e74c3c'] + ['#95a5a6'] * 4
bars = ax.bar(method_names, method_r, color=bar_colors, edgecolor='black', lw=0.5, alpha=0.85)
for b, v in zip(bars, method_r):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
            f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Pearson r', fontsize=12)
ax.set_title('S669 Benchmark Comparison', fontsize=13, fontweight='bold')
ax.set_ylim(0, 0.7)
ax.grid(True, alpha=0.3, axis='y')

# Panel C: Learned distance kernels
ax = axes[2]
model.load_state_dict(results['Full']['state'])
model.eval()
for layer_idx, (layer_name, layer) in enumerate([('Layer 1', model.gat1), ('Layer 2', model.gat2)]):
    if hasattr(layer, 'dk'):
        dk = layer.dk
        sigmas = torch.exp(dk.log_s).detach().cpu().numpy()
        weights = F.softmax(dk.w, dim=0).detach().cpu().numpy()
        d_norm = np.linspace(0, 1, 200)
        d_angstrom = d_norm * 12.0
        kernel_labels = ['Short-range', 'Medium-range', 'Long-range']
        kernel_colors = ['#e74c3c', '#3498db', '#2ecc71']
        for k in range(dk.n_k):
            response = weights[k] * np.exp(-d_norm ** 2 / (2 * sigmas[k] ** 2 + 1e-8))
            linestyle = '-' if layer_idx == 0 else '--'
            label = (f'{kernel_labels[k]} (s={sigmas[k]*12:.1f}A, w={weights[k]:.2f})'
                     if layer_idx == 0 else None)
            ax.plot(d_angstrom, response, linestyle, color=kernel_colors[k],
                    lw=2, alpha=0.7, label=label)

ax.set_xlabel('Distance (A)', fontsize=12)
ax.set_ylabel('Attention Bias', fontsize=12)
ax.set_title('Learned Distance Kernels', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 12)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'results_v4.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save numeric results
summary = {name: {k: float(v) for k, v in r.items() if k in ['pcc', 'scc', 'rmse', 'mae']}
           for name, r in results.items()}
with open(os.path.join(RESULTS_DIR, 'results_v4.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Results saved to {RESULTS_DIR}")